In [ ]:
# !pip install ultralytics

In [ ]:
import numpy as np
from dataclasses import dataclass
import os
import math
import shutil
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary as torch_summary
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
import wandb
import time
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

%matplotlib inline

In [ ]:
@dataclass
class Config:
    patches_yolo_config_yaml: str
    original_yolo_config_yaml: str
    patches_train_images_folder: str
    patches_train_labels_folder: str
    patches_val_images_folder: str
    patches_val_labels_folder: str
    original_val_images_folder: str
    original_val_labels_folder: str
    training_output_folder: str
    saved_weights_filepath: str
    video_input_filepath: str
    video_output_filepath: str
    device: str

    # noinspection PyAttributeOutsideInit
    def init(self, training):
        self.training = training
        if self.training:
            os.makedirs(self.training_output_folder, exist_ok=True)

        self.seed = 8675309
        self.batch_size = 32
        self.starting_learning_rate = 3e-4
        self.max_epochs = 200
        self.patience = 8
        self.num_workers = 8 if self.device == 'cuda' else 0
        self.pin_memory = self.num_workers > 0
        self.image_width = 640
        self.image_height = 640
        self.image_size = 640
        self.use_amp = self.device == 'cuda'
        self.verbose = False

        self.patches_val_ids = np.array([Path(f).stem for f in os.listdir(config.patches_val_images_folder)])
        self.original_val_ids = np.array([Path(f).stem for f in os.listdir(config.original_val_images_folder)])

        self.imagenet_mean_cpu_tensor = torch.tensor(imagenet_mean_array)
        self.imagenet_std_cpu_tensor = torch.tensor(imagenet_std_array)
        self.channelwise_imagenet_mean_cpu_tensor = self.imagenet_mean_cpu_tensor.view(3, 1, 1)
        self.channelwise_imagenet_std_cpu_tensor = self.imagenet_std_cpu_tensor.view(3, 1, 1)
        self.imagenet_mean_gpu_tensor = gpu_tensor(imagenet_mean_array)
        self.imagenet_std_gpu_tensor = gpu_tensor(imagenet_std_array)
        self.channelwise_imagenet_mean_gpu_tensor = self.imagenet_mean_gpu_tensor.view(3, 1, 1)
        self.channelwise_imagenet_std_gpu_tensor = self.imagenet_std_gpu_tensor.view(3, 1, 1)

        self.model_name = 'yolo26m.pt'


config: Config = None
""" Set to environment-relevant config before training/inference """;

In [ ]:
local_config = Config(
    patches_yolo_config_yaml='data_patches/data.yaml',
    original_yolo_config_yaml='data/data.yaml',
    patches_train_images_folder='data_patches/train/images/',
    patches_train_labels_folder='data_patches/train/labels/',
    patches_val_images_folder='data_patches/val/images/',
    patches_val_labels_folder='data_patches/val/labels/',
    original_val_images_folder='data/val/images/',
    original_val_labels_folder='data/val/labels/',
    training_output_folder='data_gen/',
    saved_weights_filepath='data_gen/best.pt',
    video_input_filepath='data/inference_traffic_light_video.mp4',
    video_output_filepath='data_gen/traffic_light_detection_output_video.mp4',
    device='cpu',
)
kaggle_config = Config(
    patches_yolo_config_yaml='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/data.yaml',
    original_yolo_config_yaml='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/data.yaml',
    patches_train_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/train/images/',
    patches_train_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/train/labels/',
    patches_val_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/val/images/',
    patches_val_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/val/labels/',
    original_val_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/val/images/',
    original_val_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/val/labels/',
    training_output_folder='/kaggle/working/',
    saved_weights_filepath='/kaggle/input/datasets/kyledunne/traffic-lights-yolo-best-weights/best.pt',
    video_input_filepath='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/inference_traffic_light_video.mp4',
    video_output_filepath='/kaggle/working/traffic_light_detection_output_video.mp4',
    device='cuda',
)

In [ ]:
imagenet_mean_tuple = (0.485, 0.456, 0.406)
imagenet_std_tuple = (0.229, 0.224, 0.225)
imagenet_mean_array = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std_array = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def gpu_tensor(numpy_array):
    return torch.tensor(numpy_array, device=config.device)

def gpu_image_tensor_to_numpy_array(image_tensor):
    image = denormalize(image_tensor, config.channelwise_imagenet_mean_gpu_tensor, config.channelwise_imagenet_std_gpu_tensor)
    image = torch.clamp(image, 0, 1)
    image = image.permute(1, 2, 0).cpu().numpy()
    return (image * 255).astype(np.uint8)

def normalize(tensor, mean, std):
    return (tensor - mean) / std

def denormalize(tensor, mean, std):
    return tensor * std + mean

def print_model_torchinfo(model: nn.Module):
    print(torch_summary(model, input_size=(1, 3, config.image_width, config.image_height)))

def print_model(model: nn.Module):
    for name, module in model.named_modules():
        print(name, "->", module.__class__.__name__)

def create_dataloader(dataset, shuffle):
    return DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle, num_workers=config.num_workers, pin_memory=config.pin_memory, generator=config.generator)

def _num_batches(dataloader):
    return math.ceil(len(dataloader.dataset) / config.batch_size)

CLASS_LABELS = ['green', 'off', 'red', 'wait_on', 'yellow']

In [ ]:
def plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=None,
        pred_bounding_boxes_class_labels=None,
        pred_boxes_confidence=None,
        true_bounding_boxes=None,
        true_bounding_boxes_class_labels=None,
        white_text_background=True,
):
    for i, image in enumerate(images):
        h, w = image.shape[:2]
        fig, ax = plt.subplots(1)
        ax.imshow(image)
        ax.axis('off')

        if true_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(true_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='orange', facecolor='none')
                ax.add_patch(rect)
                if true_bounding_boxes_class_labels is not None:
                    label = true_bounding_boxes_class_labels[i][j]
                    if white_text_background:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left')

        if pred_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(pred_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                if pred_boxes_confidence is not None:
                    conf_str = f"{int(pred_boxes_confidence[i][j] * 100)}%"
                    if pred_bounding_boxes_class_labels is not None:
                        label = f"{pred_bounding_boxes_class_labels[i][j]} {conf_str}"
                    else:
                        label = conf_str
                    if white_text_background:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left')

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_val_images_with_ground_truth_bounding_boxes(num_images_to_show=3, white_text_background=True):
    ids = np.random.choice(config.original_val_ids, num_images_to_show)
    images = []
    all_boxes = []
    all_class_labels = []
    for image_id in ids:
        image_path = f'{config.original_val_images_folder}{image_id}.jpg'
        image_label = f'{config.original_val_labels_folder}{image_id}.txt'
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        images.append(image)
        boxes = []
        box_labels = []
        with open(image_label, 'r') as label_file:
            for line in label_file:
                line_segments = line.strip().split()
                box_labels.append(CLASS_LABELS[int(line_segments[0])])
                coords = [float(c) for c in line_segments[-4:]]
                boxes.append(coords)
        all_boxes.append(boxes)
        all_class_labels.append(box_labels)
    plot_images_with_bounding_boxes(images, true_bounding_boxes=all_boxes, true_bounding_boxes_class_labels=all_class_labels, white_text_background=white_text_background)

In [ ]:
def train_yolo():
    start_time = time.time()
    print(f't=0: Initializing training setup...')

    # Initialize wandb run (logs hyperparams and enables automatic metric logging)
    wandb.init(
        project='traffic-light-detection',
        name=f'run={int(start_time)}',
        config={
            'model': config.model_name,
            'epochs': config.max_epochs,
            'batch_size': config.batch_size,
            'image_size': config.image_size,
            'lr0': config.starting_learning_rate,
            'patience': config.patience,
            'seed': config.seed,
            'amp': config.use_amp,
            'device': config.device,
        }
    )

    model = YOLO(config.model_name)

    # --- wandb metric logging via Ultralytics callbacks ---
    def on_train_epoch_end(trainer):
        """Log training losses and learning rates after each training epoch."""
        epoch = trainer.epoch + 1
        loss_items = trainer.label_loss_items(trainer.tloss, prefix='train')
        metrics = {}
        for key in ('train/box_loss', 'train/cls_loss', 'train/dfl_loss'):
            if key in loss_items:
                metrics[key] = loss_items[key]
        for key in ('lr/pg0', 'lr/pg1', 'lr/pg2'):
            if key in trainer.lr:
                metrics[key] = trainer.lr[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    def on_fit_epoch_end(trainer):
        """Log validation metrics after each validation pass."""
        epoch = trainer.epoch + 1
        metrics = {}
        for key in ('metrics/precision(B)', 'metrics/recall(B)',
                     'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
                     'val/box_loss', 'val/cls_loss', 'val/dfl_loss'):
            if key in trainer.metrics:
                metrics[key] = trainer.metrics[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    model.add_callback('on_train_epoch_end', on_train_epoch_end)
    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

    # Train — verbose=True enables per-epoch cell output logging
    results = model.train(
        data=config.patches_yolo_config_yaml,
        epochs=config.max_epochs,
        imgsz=config.image_size,
        batch=config.batch_size,
        optimizer='AdamW',
        lr0=config.starting_learning_rate,
        warmup_bias_lr=0.01,
        cos_lr=True,
        patience=config.patience,
        seed=config.seed,
        amp=config.use_amp,
        workers=config.num_workers,
        device=config.device,
        verbose=True,
        plots=True,
    )

    # Copy best model weights to config.training_output_folder
    best_src = Path(results.save_dir) / 'weights' / 'best.pt'
    os.makedirs(config.training_output_folder, exist_ok=True)
    best_dst = Path(config.training_output_folder) / 'best.pt'
    shutil.copy(str(best_src), str(best_dst))
    print(f'Best model saved to: {best_dst}')

    # Upload best model to wandb as an artifact (replaces add_wandb_callback checkpointing,
    # which is broken in kaggle environment due to unresolved imports in the callback module)
    artifact = wandb.Artifact(name='best-model', type='model')
    artifact.add_file(str(best_dst))
    wandb.log_artifact(artifact)

    wandb.finish()
    return results

In [ ]:
def print_all_coco_metrics(val_results):
    """
    Run pycocotools COCOeval and print all 12 standard COCO metrics.

    Ultralytics only triggers pycocotools automatically for officially-structured
    COCO datasets (is_coco=True). This dataset uses hex-string filenames, so even
    with the COCO layout, image IDs in predictions.json would be strings while
    pycocotools requires integers — COCOeval would score zeros. Instead, we build
    the COCO ground-truth in memory and remap IDs consistently for both GT and
    predictions before running COCOeval.

    Requires model.val() to have been called with save_json=True so that
    predictions.json exists in val_results.save_dir.
    """

    pred_json_path = Path(val_results.save_dir) / 'predictions.json'
    images_dir = Path(config.original_val_images_folder)
    labels_dir = Path(config.original_val_labels_folder)

    # Load predictions (image_id is a hex string, e.g. "b01d46f9911d558b")
    with open(pred_json_path, 'r') as f:
        raw_preds = json.load(f)

    if not raw_preds:
        print('\n--- COCO Metrics (pycocotools) ---')
        print('  No predictions found — skipping COCOeval.')
        return

    # pycocotools requires integer IDs. Build a stable str→int mapping from
    # all val images (not just predicted ones, so recall denominator is correct).
    all_stems = sorted(p.stem for p in images_dir.glob('*.jpg'))
    stem_to_id = {stem: i + 1 for i, stem in enumerate(all_stems)}

    # Remap predictions to integer IDs
    remapped_preds = [
        {**pred, 'image_id': stem_to_id[pred['image_id']]}
        for pred in raw_preds
        if pred['image_id'] in stem_to_id
    ]

    # Build COCO ground-truth structure in memory
    coco_images = []
    coco_annotations = []
    ann_id = 1

    for stem in all_stems:
        img = cv2.imread(str(images_dir / f'{stem}.jpg'))
        h, w = img.shape[:2] if img is not None else (640, 640)
        int_id = stem_to_id[stem]

        coco_images.append({'id': int_id, 'file_name': f'{stem}.jpg', 'width': w, 'height': h})

        label_path = labels_dir / f'{stem}.txt'
        if not label_path.exists() or label_path.stat().st_size == 0:
            continue

        with open(label_path) as lf:
            for line in lf:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_idx, xc, yc, bw, bh = map(float, parts)
                x = (xc - bw / 2) * w
                y = (yc - bh / 2) * h
                bw_px, bh_px = bw * w, bh * h
                coco_annotations.append({
                    'id': ann_id, 'image_id': int_id, 'category_id': int(cls_idx),
                    'bbox': [x, y, bw_px, bh_px], 'area': bw_px * bh_px, 'iscrowd': 0,
                })
                ann_id += 1

    # Load into pycocotools without writing any files
    coco_gt = COCO()
    coco_gt.dataset = {
        'images': coco_images,
        'annotations': coco_annotations,
        'categories': [{'id': i, 'name': name, 'supercategory': 'object'} for i, name in enumerate(CLASS_LABELS)],
    }
    coco_gt.createIndex()
    coco_dt = coco_gt.loadRes(remapped_preds)

    # Run evaluation and print all 12 standard COCO metrics
    print('\n--- COCO Metrics (pycocotools) ---')
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

In [ ]:
def predict_yolo(return_preds=True, print_coco_metrics=True):
    model = YOLO(config.saved_weights_filepath)

    # Run validation — logs all available COCO metrics to stdout.
    # save_json=True saves predictions.json to val_results.save_dir, which is read
    # by print_all_coco_metrics() to compute all 12 standard COCO metrics.
    # verbose=True prints the per-class mAP table from ultralytics.
    val_results = model.val(
        data=config.original_yolo_config_yaml,
        imgsz=config.image_size,
        batch=config.batch_size,
        workers=config.num_workers,
        device=config.device,
        verbose=True,
        plots=True,
        save_json=True,
    )

    if print_coco_metrics:
        print_all_coco_metrics(val_results)

    if not return_preds:
        return

    # Run per-image inference on the val set to collect bounding boxes for visualization.
    # model.val() doesn't expose per-image Results objects, so model.predict() is used.
    predictions = model.predict(
        source=config.original_val_images_folder,
        imgsz=config.image_size,
        device=config.device,
        verbose=False,
    )

    return predictions

In [ ]:
def predict_yolo_with_visualizations(num_to_visualize=3, print_coco_metrics=False):
    predictions = predict_yolo(print_coco_metrics=print_coco_metrics)

    indices = np.random.choice(len(predictions), num_to_visualize, replace=False)
    subset = [predictions[i] for i in indices]

    images = [cv2.cvtColor(r.orig_img, cv2.COLOR_BGR2RGB) for r in subset]
    pred_bounding_boxes = [r.boxes.xywhn.cpu().numpy().tolist() for r in subset]
    pred_boxes_confidence = [r.boxes.conf.cpu().numpy().tolist() for r in subset]
    pred_class_labels = [[CLASS_LABELS[int(c)] for c in r.boxes.cls.cpu().numpy().tolist()] for r in subset]

    plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=pred_bounding_boxes,
        pred_bounding_boxes_class_labels=pred_class_labels,
        pred_boxes_confidence=pred_boxes_confidence,
    )

In [ ]:
def predict_video():
    model = YOLO(config.saved_weights_filepath)

    video = cv2.VideoCapture(config.video_input_filepath)
    if not video.isOpened():
        print('Error opening video file')
        return

    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)

    out_dir = os.path.dirname(config.video_output_filepath)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    output_file = cv2.VideoWriter(
        filename=config.video_output_filepath,
        fourcc=cv2.VideoWriter_fourcc(*'mp4v'),
        fps=float(fps),
        frameSize=(width, height),
        isColor=True,
    )

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        results = model.predict(
            source=frame,
            imgsz=config.image_size,
            device=config.device,
            verbose=False,
        )

        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()  # [N, 4] pixel coords (x1, y1, x2, y2)
            confs = result.boxes.conf.cpu().numpy()  # [N]
            classes = result.boxes.cls.cpu().numpy()  # [N]
            for (x1, y1, x2, y2), conf, cls_idx in zip(boxes, confs, classes):
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color=(0, 165, 255), thickness=2)
                label = f'{CLASS_LABELS[int(cls_idx)]} {int(conf * 100)}%'
                (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1, y1 - th - baseline - 5), (x1 + tw, y1), color=(255, 255, 255), thickness=-1)
                cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                            fontScale=0.6, color=(0, 165, 255), thickness=2)

        output_file.write(frame)

    video.release()
    output_file.release()
    print(f'Video saved to: {config.video_output_filepath}')

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret('wandb_key')
# !wandb login $wandb_key

In [ ]:
config = local_config
config.init(training=False)
plot_val_images_with_ground_truth_bounding_boxes(white_text_background=False)